# DRI-25 Offline llama.cpp Validation

This notebook is for the local laptop proof, not Colab. Run the setup/pre-stage cells while online, then turn off Wi-Fi / enable airplane mode before the final validation cell.

The acceptance run uses only local files: text GGUF, Qwen2-VL `mmproj` GGUF, `llama-mtmd-cli`, and one X-ray image. It saves latency, tokens/sec, raw output, and a model-card-ready metrics snippet.

In [ ]:
from pathlib import Path

PROJECT_REPO = "https://github.com/ShivamSinghNow/Drishti.git"
BRANCH = "codex/dri25-offline-llamacpp-validation"
PROJECT_DIR = Path.cwd()

LLAMA_CPP_DIR = PROJECT_DIR / "external" / "llama.cpp"
GGUF_DIR = PROJECT_DIR / "outputs" / "dri24-gguf"
TEXT_GGUF = GGUF_DIR / "drishti-qwen2vl-run4-q4_k_m.gguf"
MMPROJ_GGUF = GGUF_DIR / "mmproj-drishti-qwen2vl-run4-f16.gguf"
SAMPLE_IMAGE = PROJECT_DIR / "outputs" / "dri25-offline-llamacpp" / "sample_xray.png"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "dri25-offline-llamacpp"

print(PROJECT_DIR)


In [ ]:
import subprocess
import sys

def run(command: list[str], *, cwd: Path | None = None, capture: bool = False) -> subprocess.CompletedProcess | None:
    print("\n$ " + " ".join(command))
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        text=True,
        capture_output=capture,
    )
    if capture:
        print(result.stdout)
        print(result.stderr)
    result.check_returncode()
    return result if capture else None


## Online Pre-Stage

Use this before the offline proof. It updates the repo and makes sure the local validation CLI is available.

In [ ]:
run(["git", "fetch", "origin", BRANCH], cwd=PROJECT_DIR)
run(["git", "checkout", BRANCH], cwd=PROJECT_DIR)
run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=PROJECT_DIR)


## Build llama.cpp If Needed

Skip this if `external/llama.cpp/build/bin/llama-mtmd-cli` already exists from DRI-24.

In [ ]:
if not LLAMA_CPP_DIR.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_CPP_DIR)])
else:
    run(["git", "pull", "--ff-only"], cwd=LLAMA_CPP_DIR)

run(["cmake", "-B", "build", "-DLLAMA_CURL=OFF"], cwd=LLAMA_CPP_DIR)
run(["cmake", "--build", "build", "--config", "Release", "-j", "4", "--target", "llama-mtmd-cli"], cwd=LLAMA_CPP_DIR)


## Ensure Local Artifacts Exist

These files should already be present from DRI-24. If they are not, run the DRI-24 GGUF export notebook first while online.

In [ ]:
required_files = {
    "text_gguf": TEXT_GGUF,
    "mmproj_gguf": MMPROJ_GGUF,
}

for name, path in required_files.items():
    print(name, path, path.exists())
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")


## Pick A Local X-Ray

If `data/processed/val.jsonl` exists, this cell copies the first `active_tb` validation X-ray into the offline output folder. Otherwise set `SAMPLE_IMAGE` manually to any local chest X-ray PNG.

In [ ]:
import json
import shutil

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
val_jsonl = PROJECT_DIR / "data" / "processed" / "val.jsonl"

if not SAMPLE_IMAGE.exists() and val_jsonl.exists():
    with val_jsonl.open(encoding="utf-8") as handle:
        for line in handle:
            payload = json.loads(line)
            label = payload["messages"][1]["content"].removeprefix("Classification: ").strip()
            if label != "active_tb":
                continue
            for item in payload["messages"][0]["content"]:
                if item.get("type") == "image" and item.get("path"):
                    candidate = PROJECT_DIR / item["path"]
                    if candidate.exists():
                        shutil.copyfile(candidate, SAMPLE_IMAGE)
                        break
            if SAMPLE_IMAGE.exists():
                break

print(SAMPLE_IMAGE, SAMPLE_IMAGE.exists())
if not SAMPLE_IMAGE.exists():
    raise FileNotFoundError("Set SAMPLE_IMAGE to a local X-ray before the offline run.")


## Offline Acceptance Run

Turn off Wi-Fi / enable airplane mode now. This cell fails immediately if it can still reach the network check host. It also fails if the command takes more than 10 seconds or the output is not `Classification: <label>`.

In [ ]:
run([
    sys.executable,
    "verify_llamacpp_offline.py",
    "--model", str(TEXT_GGUF),
    "--mmproj", str(MMPROJ_GGUF),
    "--image", str(SAMPLE_IMAGE),
    "--llama-cpp-dir", str(LLAMA_CPP_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--require-offline",
    "--cpu-only",
    "--latency-threshold-seconds", "10",
    "--fail-on-gate-fail",
], cwd=PROJECT_DIR)


## Inspect Saved Proof

In [ ]:
import json

report_path = OUTPUT_DIR / "offline_llamacpp_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report["result"], indent=2))
print((OUTPUT_DIR / "model_card_metrics.md").read_text(encoding="utf-8"))
